# Task 3 — Query Optimization: Adaptive Query Execution (AQE)

## Overview

This notebook tests Adaptive Query Execution (AQE) using Query 4 (zone_weather_demand_variation), unmodified. Unlike the other three techniques, AQE doesn't change the SQL at all. It's a runtime setting (spark.sql.adaptive.enabled) that lets Spark adjust a query's plan during execution, once it sees the real size of intermediate results, instead of relying only on estimates made before execution starts.

Query 4 was chosen because it looked like the most structurally complex query in the library. That turned out to be the wrong reason to pick it for testing AQE, as explained in the discussion below.

## Query design

Both versions run the identical SQL. Only spark.sql.adaptive.enabled differs:

- Baseline: AQE disabled. Spark commits to one plan before execution.
- Optimized: AQE enabled, the platform's default. Spark can revise partition counts, join strategy, or skew handling once real sizes are known.

## Methodology

One warm-up run is discarded per configuration, five repetitions are timed, and the median is reported. Since spark.sql.adaptive.enabled is a runtime config, both configurations are measured in one SparkSession. The setting is changed right before each configuration's measurements and left unchanged until that configuration is done.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from data_intensive_computing.benchmark_helpers import (
    build_spark_session,
    register_delta_tables,
    timed_run,
    capture_explain,
    append_benchmark_record,
    find_project_root,
)

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "task3_benchmark_results.jsonl"

spark = build_spark_session(app_name="Task3-AQE")
register_delta_tables(spark, PROCESSED_DIR)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-501a316e-a63c-450e-ad1c-115d29c19c36;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Registered taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Registered weather: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/weather
Registered air_quality: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/air_quality
Registered taxi_zones: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_zones
Registered integrated_taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips


26/09/19 12:13:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Query

Query 4 (`zone_weather_demand_variation`) from the Task 1/2 library,
unmodified.

In [2]:
QUERY_SQL = """
WITH classified_weather_hours AS (
    SELECT
        observation_timestamp AS weather_hour,
        CASE
            WHEN coco IN (1, 2) THEN 'clear_or_fair'
            WHEN coco IN (3, 4) THEN 'cloudy'
            WHEN coco IN (5, 6) THEN 'fog'
            WHEN coco IN (7, 8, 9, 10, 11, 17, 18) THEN 'rain'
            WHEN coco IN (12, 13, 14, 15, 16, 19, 20, 21, 22) THEN 'snow_or_sleet'
            WHEN coco BETWEEN 23 AND 27 THEN 'storm'
            ELSE 'unknown'
        END AS weather_condition
    FROM weather
    WHERE observation_timestamp IS NOT NULL
), trip_bounds AS (
    SELECT MIN(pickup_hour) AS first_hour, MAX(pickup_hour) AS last_hour
    FROM taxi_trips
), relevant_weather_hours AS (
    SELECT w.weather_hour, w.weather_condition
    FROM classified_weather_hours AS w
    CROSS JOIN trip_bounds AS b
    WHERE w.weather_hour BETWEEN b.first_hour AND b.last_hour
      AND w.weather_condition <> 'unknown'
), zone_hour_counts AS (
    SELECT
        pickup_location_id AS location_id,
        pickup_hour,
        COUNT(trip_id) AS trip_count
    FROM taxi_trips
    GROUP BY pickup_location_id, pickup_hour
), zone_weather_hour_grid AS (
    SELECT
        z.location_id,
        z.borough,
        z.zone,
        w.weather_condition,
        w.weather_hour,
        COALESCE(c.trip_count, 0) AS trip_count
    FROM taxi_zones AS z
    CROSS JOIN relevant_weather_hours AS w
    LEFT JOIN zone_hour_counts AS c
        ON z.location_id = c.location_id
       AND w.weather_hour = c.pickup_hour
), zone_weather_average AS (
    SELECT
        location_id,
        borough,
        zone,
        weather_condition,
        AVG(trip_count) AS average_hourly_demand
    FROM zone_weather_hour_grid
    GROUP BY location_id, borough, zone, weather_condition
), variation AS (
    SELECT
        location_id,
        borough,
        zone,
        COUNT(*) AS weather_categories,
        MIN(average_hourly_demand) AS minimum_weather_demand,
        MAX(average_hourly_demand) AS maximum_weather_demand,
        MAX(average_hourly_demand) - MIN(average_hourly_demand) AS demand_range,
        STDDEV_POP(average_hourly_demand) AS demand_stddev
    FROM zone_weather_average
    GROUP BY location_id, borough, zone
    HAVING COUNT(*) >= 2
), ranked AS (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY demand_range DESC) AS variation_rank
    FROM variation
)
SELECT
    variation_rank,
    location_id,
    borough,
    zone,
    weather_categories,
    ROUND(minimum_weather_demand, 3) AS minimum_average_hourly_demand,
    ROUND(maximum_weather_demand, 3) AS maximum_average_hourly_demand,
    ROUND(demand_range, 3) AS average_hourly_demand_range,
    ROUND(demand_stddev, 3) AS average_hourly_demand_stddev
FROM ranked
WHERE variation_rank <= 20
ORDER BY variation_rank, location_id
"""

## Baseline: AQE disabled

In [3]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

baseline_timing = timed_run(spark, "zone_weather_demand_variation_aqe_off", QUERY_SQL, repetitions=5)
print("Baseline (AQE disabled):", baseline_timing)

26/09/19 12:13:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Baseline (AQE disabled): {'query_name': 'zone_weather_demand_variation_aqe_off', 'row_count': 20, 'repetitions': 5, 'all_times_seconds': [1.1509, 0.9158, 0.873, 0.9474, 0.7818], 'median_seconds': 0.9158, 'min_seconds': 0.7818, 'max_seconds': 1.1509}


In [4]:
baseline_plan = capture_explain(spark.sql(QUERY_SQL), mode="formatted")
baseline_rows = spark.sql(QUERY_SQL).collect()

print("=" * 30, "BASELINE PLAN (AQE disabled)", "=" * 30)
print(baseline_plan)

26/09/19 12:13:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


============================== BASELINE PLAN (AQE disabled) ==============================
== Physical Plan ==
* Sort (46)
+- * Project (45)
   +- * Filter (44)
      +- Window (43)
         +- WindowGroupLimit (42)
            +- * Sort (41)
               +- Exchange (40)
                  +- WindowGroupLimit (39)
                     +- * Sort (38)
                        +- * Filter (37)
                           +- * HashAggregate (36)
                              +- Exchange (35)
                                 +- * HashAggregate (34)
                                    +- * HashAggregate (33)
                                       +- Exchange (32)
                                          +- * HashAggregate (31)
                                             +- * Project (30)
                                                +- * SortMergeJoin LeftOuter (29)
                                                   :- * Sort (20)
                                                   :  +- 

## Optimized: AQE enabled

The plan is captured *after* `timed_run` has already executed this exact
query under AQE, so it reflects the plan Spark actually converged on at
runtime (its `== Final Plan ==` section) rather than only the plan chosen
before any shuffle stage had materialized.

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

optimized_timing = timed_run(spark, "zone_weather_demand_variation_aqe_on", QUERY_SQL, repetitions=5)
print("Optimized (AQE enabled):", optimized_timing)

26/09/19 12:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 1

Optimized (AQE enabled): {'query_name': 'zone_weather_demand_variation_aqe_on', 'row_count': 20, 'repetitions': 5, 'all_times_seconds': [0.8821, 0.778, 0.8001, 0.8237, 0.7986], 'median_seconds': 0.8001, 'min_seconds': 0.778, 'max_seconds': 0.8821}


26/09/19 12:13:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [6]:
optimized_plan = capture_explain(spark.sql(QUERY_SQL), mode="formatted")
optimized_rows = spark.sql(QUERY_SQL).collect()

print("=" * 30, "OPTIMIZED PLAN (AQE enabled)", "=" * 30)
print(optimized_plan)

26/09/19 12:13:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 1

============================== OPTIMIZED PLAN (AQE enabled) ==============================
== Physical Plan ==
AdaptiveSparkPlan (43)
+- Sort (42)
   +- Project (41)
      +- Filter (40)
         +- Window (39)
            +- WindowGroupLimit (38)
               +- Sort (37)
                  +- Exchange (36)
                     +- WindowGroupLimit (35)
                        +- Sort (34)
                           +- Filter (33)
                              +- HashAggregate (32)
                                 +- Exchange (31)
                                    +- HashAggregate (30)
                                       +- HashAggregate (29)
                                          +- Exchange (28)
                                             +- HashAggregate (27)
                                                +- Project (26)
                                                   +- SortMergeJoin LeftOuter (25)
                                                      :- Sort (17)
   

26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Physical plan comparison

We compared the baseline and optimized plans directly. They are identical: no AQEShuffleRead, no difference between the Final Plan and Initial Plan, and no change in join strategy or partition counts. AQE did not adjust anything for this query.

## Verifying the window step cost

The discussion above assumed the single-partition window step was the main bottleneck, based on the warning message Spark printed, without actually confirming it. This section checks that directly using Spark's own stage-level metrics, through the same REST API the Spark UI is built on.

The query is run once more here under AQE enabled (the platform's default). The stage list before and after this run is compared to isolate exactly which stages this run created. The stage with exactly one task is the window step, since a window function with no PARTITION BY forces all data into a single partition, and Spark schedules one task per partition per stage.

In [7]:
import json
import urllib.request

def fetch_stages():
    app_id = spark.sparkContext.applicationId
    ui_url = spark.sparkContext.uiWebUrl
    stages_url = f"{ui_url}/api/v1/applications/{app_id}/stages"
    with urllib.request.urlopen(stages_url) as response:
        return json.loads(response.read())

spark.conf.set("spark.sql.adaptive.enabled", "true")

stages_before = {s["stageId"] for s in fetch_stages()}
spark.sql(QUERY_SQL).collect()
all_stages_after = fetch_stages()
new_stage_ids = {s["stageId"] for s in all_stages_after} - stages_before

new_stages = sorted(
    (s for s in all_stages_after if s["stageId"] in new_stage_ids),
    key=lambda s: s["stageId"],
)

for s in new_stages:
    print(f"stage {s['stageId']}: {s['numTasks']} task(s), "
          f"executorRunTime={s.get('executorRunTime', 0)} ms, "
          f"name={s['name'][:60]}")

total_time = sum(s.get("executorRunTime", 0) for s in new_stages)
single_task_time = sum(s.get("executorRunTime", 0) for s in new_stages if s["numTasks"] == 1)

print()
print(f"Total executorRunTime for this run: {total_time} ms")
print(f"Time in single-task stages (the window step): {single_task_time} ms")
if total_time > 0:
    print(f"Share of total: {100 * single_task_time / total_time:.1f}%")

26/09/19 12:13:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 1

stage 365: 2 task(s), executorRunTime=0 ms, name=collect at /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/
stage 366: 50 task(s), executorRunTime=199 ms, name=collect at /var/folders/_2/lsj4377d6ksfr4rqf7sn0xwm0000gp/T/
stage 367: 2 task(s), executorRunTime=0 ms, name=collect at /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/
stage 368: 50 task(s), executorRunTime=167 ms, name=collect at /var/folders/_2/lsj4377d6ksfr4rqf7sn0xwm0000gp/T/
stage 369: 1 task(s), executorRunTime=0 ms, name=collect at /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/
stage 370: 50 task(s), executorRunTime=196 ms, name=collect at /var/folders/_2/lsj4377d6ksfr4rqf7sn0xwm0000gp/T/
stage 371: 1 task(s), executorRunTime=0 ms, name=collect at /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/
stage 372: 50 task(s), executorRunTime=143 ms, name=collect at /var/folders/_2/lsj4377d6ksfr4rqf7sn0xwm0000gp/T/
stage 373: 15 task(s), executorRunTime=389 ms, name=collect at /var/folders/_2/lsj4377d6ksfr4rqf7sn0xwm0000g

26/09/19 12:13:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 12:13:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Correctness verification

Since both variants run the identical SQL text, results are collected under
each configuration and compared directly rather than through
`verify_same_results` (which assumes two different query strings run under
one fixed session state).

In [8]:
baseline_set = {tuple(r) for r in baseline_rows}
optimized_set = {tuple(r) for r in optimized_rows}

rows_only_in_before = len(baseline_set - optimized_set)
rows_only_in_after = len(optimized_set - baseline_set)
correctness_verified = (rows_only_in_before == 0) and (rows_only_in_after == 0)

print({
    "correctness_verified": correctness_verified,
    "rows_only_in_before": rows_only_in_before,
    "rows_only_in_after": rows_only_in_after,
})
assert correctness_verified, "AQE on/off produced different results — investigate before recording a result."


{'correctness_verified': True, 'rows_only_in_before': 0, 'rows_only_in_after': 0}


## Benchmark record

In [9]:
speedup = round(baseline_timing["median_seconds"] / optimized_timing["median_seconds"], 2)

has_shuffle_read = "AQEShuffleRead" in optimized_plan
has_final_initial_split = "== Final Plan ==" in optimized_plan and "== Initial Plan ==" in optimized_plan
plans_differ = False
if has_final_initial_split:
    final_section = optimized_plan.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    initial_section = optimized_plan.split("== Initial Plan ==")[1]
    plans_differ = final_section.strip() != initial_section.strip()

plan_change_summary = (
    "The optimized plan is wrapped in AdaptiveSparkPlan, absent from the baseline. "
    + ("AQEShuffleRead is present, indicating AQE adjusted shuffle partitions after "
       "observing their actual size. "
       if has_shuffle_read else
       "AQEShuffleRead is not present, indicating no post-shuffle partition "
       "adjustment was triggered at this data scale. ")
    + (f"The Final Plan differs from the Initial Plan, confirming AQE altered "
       f"execution based on runtime statistics."
       if plans_differ else
       "The Final Plan matches the Initial Plan, indicating the static estimates "
       "were already accurate enough that no re-planning occurred for this query "
       "at this data scale.")
)

record = {
    "query_name": "zone_weather_demand_variation",
    "technique": "aqe",
    "before_time_sec": baseline_timing["median_seconds"],
    "after_time_sec": optimized_timing["median_seconds"],
    "speedup": speedup,
    "plan_change_summary": plan_change_summary,
    "correctness_verified": correctness_verified,
}

append_benchmark_record(record, RESULTS_PATH)
print(f"Speedup: {speedup}x — appended to {RESULTS_PATH}")
print()
print("plan_change_summary:", plan_change_summary)

Speedup: 1.14x — appended to /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/artifacts/task3_benchmark_results.jsonl

plan_change_summary: The optimized plan is wrapped in AdaptiveSparkPlan, absent from the baseline. AQEShuffleRead is not present, indicating no post-shuffle partition adjustment was triggered at this data scale. The Final Plan matches the Initial Plan, indicating the static estimates were already accurate enough that no re-planning occurred for this query at this data scale.


## Discussion

We picked Query 4 because it looked the most complex, which turned out to be the wrong basis for choosing a query to test AQE on.

Query 4's last step is DENSE_RANK() OVER (ORDER BY demand_range DESC), with no PARTITION BY. This forces all rows into a single partition for that step. We first guessed this was the reason AQE showed no effect, since AQE cannot change how a window function partitions its data.

We checked this directly using Spark's stage-level metrics (see "Verifying the window step cost" below). The single-partition window step took 9 ms out of a 5545 ms total run, about 0.2 percent. It is not the bottleneck. Our original guess was wrong.

The real reason AQE shows no effect here is the same as for the other two queries we tested: the plan had nothing for AQE to adjust anywhere, not just in the window step. This matches the plan comparison below, which shows no AQEShuffleRead and no difference between the Final Plan and Initial Plan. We don't know what the 5545 ms is actually spent on. Query 4 chains a CROSS JOIN, a LEFT JOIN, and several GROUP BY stages before the ranking step, and any of these is a more likely candidate than the ranking step itself.

Result: 1.14x, essentially no speedup. The plan shows no AQEShuffleRead, and the Final Plan matches the Initial Plan, meaning AQE found nothing worth changing.

This query is not a good one for demonstrating what AQE can do. monthly_zone_demand and distance_by_weather are better candidates and are covered in their own notebooks.

In [10]:
spark.stop()